In [ ]:
# ============================================================
# Image Generation API Demo
# ============================================================
# Phần 1: OpenAI DALL-E 3 API (closed source, trả phí ~$0.04-0.08/ảnh)
# Phần 2: HuggingFace Inference API (miễn phí, zero GPU)

In [ ]:
# !pip install openai huggingface_hub Pillow matplotlib requests python-dotenv

In [ ]:
import os
import base64
from io import BytesIO

from huggingface_hub import InferenceClient
from openai import OpenAI

import matplotlib.pyplot as plt
from dotenv import load_dotenv
from PIL import Image

load_dotenv()

In [ ]:
# ============================================================
# PHẦN 1: OpenAI DALL-E 3 API (Closed Source)
# ============================================================
# Tạo file .env từ .env.example và điền API key:
#   cp .env.example .env
# Pricing: https://openai.com/api/pricing/

In [ ]:
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

print("OpenAI client initialized.")
print(f"API key set: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No - create .env file from .env.example!'}")

In [ ]:
# Generate ảnh với DALL-E 3

In [ ]:
prompt = "A cute golden retriever puppy playing in a field of sunflowers, sunny day, photorealistic"

response = openai_client.images.generate(
    model="dall-e-3",
    prompt=prompt,
    size="1024x1024",
    quality="standard",
    n=1,
    response_format="b64_json"
)

In [ ]:
# Decode ảnh từ base64
image_data = base64.b64decode(response.data[0].b64_json)
dalle_image = Image.open(BytesIO(image_data))

In [ ]:
# DALL-E 3 có thể chỉnh sửa prompt, xem revised_prompt
print(f"Original prompt: {prompt}")
print(f"Revised prompt: {response.data[0].revised_prompt}")

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(dalle_image)
plt.title("DALL-E 3 - Text-to-Image", fontsize=14)
plt.axis("off")
plt.show()

In [ ]:
# So sánh các size và quality khác nhau của DALL-E 3
# Size options: 1024x1024, 1024x1792 (portrait), 1792x1024 (landscape)
# Quality options: "standard", "hd"

In [ ]:
prompt = "A majestic mountain landscape at sunset with a crystal clear lake reflection"

configs = [
    {"size": "1024x1024", "quality": "standard"},
    {"size": "1024x1024", "quality": "hd"},
    {"size": "1792x1024", "quality": "standard"},
]

dalle_images = []
for config in configs:
    response = openai_client.images.generate(
        model="dall-e-3",
        prompt=prompt,
        size=config["size"],
        quality=config["quality"],
        n=1,
        response_format="b64_json"
    )
    image_data = base64.b64decode(response.data[0].b64_json)
    dalle_images.append(Image.open(BytesIO(image_data)))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for i, (img, config) in enumerate(zip(dalle_images, configs)):
    axes[i].imshow(img)
    axes[i].set_title(f"{config['size']} / {config['quality']}", fontsize=14)
    axes[i].axis("off")

plt.suptitle(f"DALL-E 3: Size & Quality comparison\nPrompt: \"{prompt}\"", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PHẦN 2: HuggingFace Inference API (Miễn phí, Zero GPU)
# ============================================================
# Không cần GPU! Chạy được từ bất kỳ laptop nào.
# HuggingFace Inference API cung cấp free tier (rate-limited)
# HF_TOKEN đã được load từ .env file ở trên

In [ ]:
hf_client = InferenceClient(
    token=os.getenv("HF_TOKEN", None)
)

print("HuggingFace InferenceClient initialized.")
print(f"HF token set: {'Yes' if os.getenv('HF_TOKEN') else 'No (anonymous, lower rate limit)'}")

In [ ]:
# Text-to-Image với FLUX.1-schnell qua HuggingFace API
# Model: black-forest-labs/FLUX.1-schnell (Apache-2.0, SOTA quality)

prompt = "A cute golden retriever puppy playing in a field of sunflowers, sunny day, photorealistic"

flux_image = hf_client.text_to_image(
    prompt=prompt,
    model="black-forest-labs/FLUX.1-schnell"
)

plt.figure(figsize=(8, 8))
plt.imshow(flux_image)
plt.title("FLUX.1 [schnell] via HuggingFace API", fontsize=14)
plt.axis("off")
plt.show()

In [ ]:
# Text-to-Image với Stable Diffusion XL qua HuggingFace API
# Model: stabilityai/stable-diffusion-xl-base-1.0

prompt = "A cute golden retriever puppy playing in a field of sunflowers, sunny day, photorealistic"

sdxl_image = hf_client.text_to_image(
    prompt=prompt,
    model="stabilityai/stable-diffusion-xl-base-1.0"
)

plt.figure(figsize=(8, 8))
plt.imshow(sdxl_image)
plt.title("Stable Diffusion XL via HuggingFace API", fontsize=14)
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# PHẦN 3: So sánh tổng hợp - OpenAI DALL-E 3 vs HuggingFace API
# ============================================================
# Cùng 1 prompt, so sánh kết quả từ 3 nguồn khác nhau:
# 1. OpenAI DALL-E 3 (closed source, trả phí)
# 2. FLUX.1-schnell via HF API (open weights, miễn phí)
# 3. SDXL via HF API (open source, miễn phí)

In [ ]:
prompt = "An astronaut riding a horse on Mars, cinematic lighting, detailed, high quality"

In [ ]:
# DALL-E 3
response = openai_client.images.generate(
    model="dall-e-3",
    prompt=prompt,
    size="1024x1024",
    quality="standard",
    n=1,
    response_format="b64_json"
)
dalle_img = Image.open(BytesIO(base64.b64decode(response.data[0].b64_json)))

In [ ]:
# FLUX.1-schnell via HF
flux_img = hf_client.text_to_image(
    prompt=prompt,
    model="black-forest-labs/FLUX.1-schnell"
)

In [ ]:
# SDXL via HF
sdxl_img = hf_client.text_to_image(
    prompt=prompt,
    model="stabilityai/stable-diffusion-xl-base-1.0"
)

In [ ]:
# Hiển thị so sánh
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

images_and_labels = [
    (dalle_img, "DALL-E 3\n(OpenAI, paid)"),
    (flux_img, "FLUX.1 schnell\n(HF API, free)"),
    (sdxl_img, "SDXL\n(HF API, free)"),
]

for i, (img, label) in enumerate(images_and_labels):
    axes[i].imshow(img)
    axes[i].set_title(label, fontsize=14)
    axes[i].axis("off")

plt.suptitle(f"Closed Source vs Open Source comparison\nPrompt: \"{prompt}\"", fontsize=16)
plt.tight_layout()
plt.show()